# Model III
## Rozpoznawanie instrumentów za pomocą konwolucji (ResNet18)

### Biblioteki

In [1]:
import sys
import os
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd

import time

from model import Resnet
from trening import get_device, train_model
from przygotowanie_danych import (
    create_datasets,
    create_loaders
)

Hiperparametry modelu

In [3]:
dataset = "1_1"
model_name = "resnet18"

epochs = 100
patience = 10
batch_size = 32
learning_rate = 0.0001
dropout = 0.3

weights_type = "default"
weights_path = None

In [4]:
# Parameters
dataset = "2_6"
model_name = "resnet18"
epochs = 100
patience = 10
batch_size = 32
learning_rate = 0.0001
dropout = 0.3
weights_type = "custom"
weights_path = "wyniki\\1.pth"


In [5]:
results_file = Path("wyniki/modele.csv")

if results_file.exists():
    results = pd.read_csv(results_file)

    combination_exists = (
        (results["Dataset"] == dataset) &
        (results["Model"] == model_name) &
        (results["Batch size"] == batch_size) &
        (results["Learning rate"] == learning_rate) &
        (results["Dropout"] == dropout)
    ).any()

    if combination_exists:
        raise ValueError("Ta kombinacja datasetu i hiperparametrów była już testowana. Zatrzymuję program.")

print("Nowa kombinacja. Kontynuuję wykonywanie kodu.")

Nowa kombinacja. Kontynuuję wykonywanie kodu.


In [6]:
existing_numbers = [
    int(f[:-4])
    for f in os.listdir("wyniki")
    if f.endswith(".pth")
]

learning_id = max(existing_numbers, default=0) + 1

weights_filename = f"{learning_id}.pth"

weights_filename

'6.pth'

In [7]:
train_dataset, val_dataset, test_dataset = create_datasets(dataset)

train_loader, val_loader, test_loader = create_loaders(train_dataset, val_dataset, test_dataset, batch_size=batch_size)

Wczytuję spektrogramy z: C:\Users\Wiktor\Documents\GitHub\instrument-recognition-project\spectrograms\2_6


In [8]:
device = get_device()
print(device)

model = Resnet(
    dropout=dropout,
    weights_type=weights_type,
    weights_path=weights_path
).to(device)

criterion = nn.MultiLabelSoftMarginLoss()

optimizer = optim.Adam(model.parameters(), lr=learning_rate)

cuda


Pętla treningowa

In [9]:
start_time = time.perf_counter()

model, history, best_epoch, best_val_loss = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    epochs=epochs,
    patience=patience,
    model_name= str(weights_filename)
)

training_time = time.perf_counter() - start_time

print(f"Czas treningu: {training_time / 60:.2f} min")

Epoka 1/100 | Train loss: 0.3730 | Val loss: 0.3756


Epoka 2/100 | Train loss: 0.2539 | Val loss: 0.4290


Epoka 3/100 | Train loss: 0.1908 | Val loss: 0.4068


Epoka 4/100 | Train loss: 0.1383 | Val loss: 0.4461


Epoka 5/100 | Train loss: 0.0997 | Val loss: 0.4531


Epoka 6/100 | Train loss: 0.0768 | Val loss: 0.4566


Epoka 7/100 | Train loss: 0.0629 | Val loss: 0.4141


Epoka 8/100 | Train loss: 0.0557 | Val loss: 0.4654


Epoka 9/100 | Train loss: 0.0520 | Val loss: 0.5286


Epoka 10/100 | Train loss: 0.0462 | Val loss: 0.5267


Zapisanie danych o modelu

In [ ]:
experiment = pd.DataFrame([{
    "Learning_ID": learning_id,
    "Dataset": dataset,
    "Model": model_name,
    "Batch size": batch_size,
    "Learning rate": learning_rate,
    "Dropout": dropout,
    "Best epoch": best_epoch,
    "Best val loss": best_val_loss,
    "Training time [min]": training_time / 60
}])

experiment.to_csv(
    results_file,
    mode="a",
    header=not results_file.exists(),
    index=False
)

In [ ]:
history_df = pd.DataFrame(history)

history_df.to_csv(
    f"wyniki/history_{learning_id}.csv",
    index_label="epoch"
)